# TradingAgents — FPT full-graph run and verified report download

Notebook mới mở trực tiếp trong Google Colab, trên nhánh thử nghiệm riêng. Nó giữ cấu trúc graph tiêu chuẩn của TradingAgents: đủ 4 analyst (market, social/sentiment, news, fundamentals), sau đó bull/bear research, research manager, trader, risk analysts và portfolio manager.

- SSI: nguồn giá OHLCV và chỉ báo kỹ thuật.
- Vnstock VCI: nguồn báo cáo tài chính Việt Nam khi package khả dụng.
- Gemini: model cho các agent, có fallback ngay trong mỗi lần gọi LLM, không chỉ ở bước preflight.
- Bước tải ZIP tách riêng: chạy lại ô cuối sẽ không gọi Gemini/SSI.

Không có lệnh mua/bán. Đặt key trong Colab Secrets, không đưa key vào code. Bản này không sửa main và dùng thư mục /content/TradingAgentsFresh tách biệt.


In [ ]:
# 1) Setup: isolated checkout; does not change your old /content/TradingAgents folder.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/PhamHuuQan/TradingAgents.git"
CODE_BRANCH = "feature/vietnam-market-foundation"
REPO_DIR = Path("/content/TradingAgentsFresh")

if not (REPO_DIR / ".git").exists():
    if REPO_DIR.exists():
        import shutil
        shutil.rmtree(REPO_DIR)
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", CODE_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", CODE_BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "checkout", "-B", CODE_BRANCH, "FETCH_HEAD"],
        check=True,
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"],
    cwd=str(REPO_DIR),
    check=True,
)

# Vnstock is an optional current-date-only provider for Vietnamese statements.
# If this optional install fails, the graph still runs and reports unavailable data.
try:
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "--extra-index-url", "https://vnstocks.com/api/packages", "vnstock",
        ],
        check=True,
    )
    print("Vnstock optional package installed.")
except subprocess.CalledProcessError:
    print("WARNING: Vnstock install failed; fundamentals tools will report unavailable data rather than invent values.")
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)

print(f"Repository ready: {CODE_BRANCH}")
print("Setup did not call Gemini or SSI.")


In [ ]:
# 2) Credentials and read-only SSI integration check. No order/trading endpoint is called.
from datetime import datetime, timedelta
from getpass import getpass
from zoneinfo import ZoneInfo

def read_colab_secret(name):
    current = os.environ.get(name)
    if isinstance(current, str) and current.strip():
        return current.strip()
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:
        value = None
    return value.strip() if isinstance(value, str) and value.strip() else None

def get_secret(name):
    return read_colab_secret(name) or getpass(f"{name} (input hidden): ")

ssi_key = get_secret("SSI_API_KEY")
ssi_secret = get_secret("SSI_API_SECRET")
if not ssi_key or not ssi_secret:
    raise ValueError("SSI_API_KEY and SSI_API_SECRET are required; values are never printed.")
os.environ["SSI_API_KEY"] = ssi_key
os.environ["SSI_API_SECRET"] = ssi_secret
del ssi_key, ssi_secret

VN_TODAY = datetime.now(ZoneInfo("Asia/Ho_Chi_Minh")).date()
FROM_DATE = (VN_TODAY - timedelta(days=30)).isoformat()
TO_DATE = VN_TODAY.isoformat()

from tradingagents.dataflows.vietnam_market.smoke import main as ssi_smoke_main
from tradingagents.dataflows.config import set_config
from tradingagents.dataflows.router import get_vendor
from tradingagents.agents.tools import get_stock_data, get_indicators

print(f"[1/2] SSI REST read-only check: FPT, {FROM_DATE} to {TO_DATE}")
exit_code = ssi_smoke_main([
    "--symbol", "FPT", "--from-date", FROM_DATE, "--to-date", TO_DATE, "--page-size", "100"
])
if exit_code != 0:
    raise RuntimeError("SSI REST check failed. Use the sanitized status message above; do not share credentials.")

set_config({"data_vendors": {"core_stock_apis": "ssi", "technical_indicators": "ssi"}})
price_vendor = get_vendor("core_stock_apis", "get_stock_data")
indicator_vendor = get_vendor("technical_indicators", "get_indicators")
print(f"[2/2] Router: prices={price_vendor}; indicators={indicator_vendor}")
if price_vendor.split(",")[0].strip() != "ssi" or indicator_vendor.split(",")[0].strip() != "ssi":
    raise RuntimeError("Router did not select SSI for both prices and indicators.")

price_result = get_stock_data.func("FPT", FROM_DATE, TO_DATE, TO_DATE)
if not isinstance(price_result, str) or "SSI FastConnect V3 read-only OHLCV for FPT" not in price_result:
    raise RuntimeError("Agent price tool did not return an SSI-sourced result.")
price_rows = [
    line for line in price_result.splitlines()
    if line.strip() and not line.startswith(("#", "Date,"))
]
if not price_rows:
    raise RuntimeError("SSI price tool returned no OHLCV rows.")

indicator_result = get_indicators.func("FPT", "rsi", TO_DATE, 30, TO_DATE)
if not isinstance(indicator_result, str) or "SSI FastConnect V3 indicator rsi for FPT" not in indicator_result:
    raise RuntimeError("Agent indicator tool did not return an SSI-sourced result.")
indicator_rows = [
    line for line in indicator_result.splitlines()
    if line.strip() and not line.startswith(("#", "Date,"))
]
if not indicator_rows:
    raise RuntimeError("SSI RSI tool returned no points.")

print(f"SSI + router + agent tools passed: {len(price_rows)} OHLCV rows; {len(indicator_rows)} RSI points.")
print("No credentials were printed or written into notebook cells.")


## 3) Kiểm tra fallback offline trước, rồi chạy AI

Ô ngay sau đây chạy 4 tình huống giả lập Gemini bằng fake model: 503 chuyển model, structured output fallback, lỗi 400 không bị che giấu, và tất cả model hỏng thì báo lỗi rõ. **Ô này không đọc key và không gọi mạng.** Chỉ khi thấy `OFFLINE SELF-TEST PASSED: 4/4 cases` thì chuyển sang ô kế tiếp.
,
,Ô phân tích thật sẽ tự kiểm tra Gemini rồi chạy toàn bộ graph với 4 analyst, research/debate, trader, risk và portfolio manager. Nếu Gemini 503 ngay giữa lượt, lớp fallback sẽ đổi model ở chính lời gọi đó. Đây là ô có thể tiêu thụ hạn mức Gemini; chạy lại ô cuối để tải ZIP thì không gọi AI.


In [ ]:
# 3) OFFLINE SELF-TEST — no credentials, no network, no Gemini/SSI calls.
# This must pass before the next cell is allowed to make any model request.
import re
import time
from langchain_core.runnables import Runnable

def is_transient_gemini_error(exc):
    value = (type(exc).__name__ + " " + str(exc)).upper()
    return any(token in value for token in (
        "503", "UNAVAILABLE", "429", "RESOURCE_EXHAUSTED",
        "500", "502", "504", "INTERNAL", "DEADLINE_EXCEEDED",
        "SERVICE UNAVAILABLE", "TIMED OUT", "SERVERERROR",
    ))

def _gemini_status(exc):
    for attr in ("status_code", "code"):
        value = getattr(exc, attr, None)
        if value is not None:
            return str(value)
    match = re.search(r"\b(408|429|500|502|503|504)\b", str(exc))
    return match.group(1) if match else type(exc).__name__

class GeminiFallbackRunnable(Runnable):
    """Fallback adapter tested offline with fake model objects."""
    def __init__(self, models, model_ids, shared_state):
        super().__init__()
        if not models or len(models) != len(model_ids):
            raise ValueError("Gemini fallback model list is empty or inconsistent.")
        self.models = list(models)
        self.model_ids = list(model_ids)
        self.shared_state = shared_state

    def invoke(self, input, config=None, **kwargs):
        start = int(self.shared_state.get("active_index", 0))
        start = max(0, min(start, len(self.models) - 1))
        for index in range(start, len(self.models)):
            try:
                output = self.models[index].invoke(input, config=config, **kwargs)
                self.shared_state["active_index"] = index
                return output
            except Exception as exc:
                if not is_transient_gemini_error(exc):
                    raise
                if index >= len(self.models) - 1:
                    raise RuntimeError(
                        f"All Gemini runtime fallback models failed; "
                        f"last_model={self.model_ids[index]}, status={_gemini_status(exc)}."
                    ) from None
                self.shared_state["active_index"] = index + 1
                print(
                    f"Offline-test fallback: {self.model_ids[index]} "
                    f"(status={_gemini_status(exc)}) -> {self.model_ids[index + 1]}"
                )
                time.sleep(0.01)
        raise RuntimeError("Gemini fallback chain exhausted.")

    def bind_tools(self, tools, **kwargs):
        bound = [model.bind_tools(tools, **kwargs) for model in self.models]
        return GeminiFallbackRunnable(bound, self.model_ids, self.shared_state)

    def with_structured_output(self, schema, **kwargs):
        bound = [model.with_structured_output(schema, **kwargs) for model in self.models]
        return GeminiFallbackRunnable(bound, self.model_ids, self.shared_state)

class _FakeGemini:
    def __init__(self, name, failures=()):
        self.name = name
        self.failures = list(failures)
        self.calls = 0
    def invoke(self, input, config=None, **kwargs):
        self.calls += 1
        if self.failures:
            raise RuntimeError(self.failures.pop(0))
        return f"OK:{self.name}"
    def bind_tools(self, tools, **kwargs):
        return self
    def with_structured_output(self, schema, **kwargs):
        return self

# 1) Tool-calling invocation falls back on 503.
state = {"active_index": 0}
primary = _FakeGemini("primary", ["503 UNAVAILABLE"])
backup = _FakeGemini("backup")
chain = GeminiFallbackRunnable([primary, backup], ["primary", "backup"], state)
assert chain.bind_tools([]).invoke("test") == "OK:backup"
assert (primary.calls, backup.calls, state["active_index"]) == (1, 1, 1)

# 2) Structured output has the same fallback behavior.
state2 = {"active_index": 0}
structured = GeminiFallbackRunnable(
    [_FakeGemini("one", ["503 UNAVAILABLE"]), _FakeGemini("two")],
    ["one", "two"],
    state2,
)
assert structured.with_structured_output(dict).invoke("test") == "OK:two"
assert state2["active_index"] == 1

# 3) Non-transient 400 must not be hidden by a model switch.
state3 = {"active_index": 0}
bad_request = _FakeGemini("primary", ["400 INVALID_ARGUMENT"])
unused = _FakeGemini("backup")
chain3 = GeminiFallbackRunnable([bad_request, unused], ["primary", "backup"], state3)
try:
    chain3.invoke("test")
except RuntimeError as exc:
    assert "400 INVALID_ARGUMENT" in str(exc)
else:
    raise AssertionError("A non-transient 400 should be raised, not silently switched.")
assert unused.calls == 0 and state3["active_index"] == 0

# 4) If every model returns 503, fail clearly and do not pretend analysis succeeded.
state4 = {"active_index": 0}
chain4 = GeminiFallbackRunnable(
    [_FakeGemini("one", ["503 UNAVAILABLE"]), _FakeGemini("two", ["503 UNAVAILABLE"])],
    ["one", "two"],
    state4,
)
try:
    chain4.invoke("test")
except RuntimeError as exc:
    assert "last_model=two, status=503" in str(exc)
else:
    raise AssertionError("Exhausted fallback chain must fail loudly.")

print("✅ OFFLINE SELF-TEST PASSED: 4/4 cases; no API key was read, no network call made.")


In [ ]:
# 3) One full TradingAgents graph run, with per-call Gemini fallback.
from copy import deepcopy
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo
import os
import re
import time

google_key = get_secret("GOOGLE_API_KEY")
if not google_key:
    raise ValueError("GOOGLE_API_KEY is empty.")
os.environ["GOOGLE_API_KEY"] = google_key
del google_key

# These stable model IDs are listed in Google's current Gemini API catalog.
MODEL_CANDIDATES = [
    "gemini-3.1-flash-lite",
    "gemini-3.5-flash-lite",
    "gemini-3.6-flash",
    "gemini-3.8-flash",
]

from langchain_google_genai import ChatGoogleGenerativeAI

def safe_gemini_error(exc):
    message = str(exc)
    key = os.environ.get("GOOGLE_API_KEY", "")
    if key:
        message = message.replace(key, "[REDACTED]")
    return message[:500]

def is_transient_gemini_error(exc):
    value = (type(exc).__name__ + " " + str(exc)).upper()
    return any(token in value for token in (
        "503", "UNAVAILABLE", "429", "RESOURCE_EXHAUSTED",
        "500", "502", "504", "INTERNAL", "DEADLINE_EXCEEDED",
        "SERVICE UNAVAILABLE", "TIMED OUT", "SERVERERROR",
    ))

def _gemini_status(exc):
    for attr in ("status_code", "code"):
        value = getattr(exc, attr, None)
        if value is not None:
            return str(value)
    match = re.search(r"\b(408|429|500|502|503|504)\b", str(exc))
    return match.group(1) if match else type(exc).__name__

# Preflight chooses the initial model; the runtime wrapper below protects later calls.
MODEL_ID = None
preflight_errors = []
for candidate in MODEL_CANDIDATES:
    try:
        ping = ChatGoogleGenerativeAI(
            model=candidate,
            google_api_key=os.environ["GOOGLE_API_KEY"],
            max_retries=0,
            max_output_tokens=24,
            timeout=45,
        ).invoke("Reply with exactly the single word READY.")
        if not getattr(ping, "content", None):
            raise RuntimeError("Gemini returned an empty response.")
        MODEL_ID = candidate
        print(f"Gemini preflight passed: {MODEL_ID}")
        break
    except Exception as exc:
        preflight_errors.append(f"{candidate}: {type(exc).__name__}, status={_gemini_status(exc)}")
        upper = (type(exc).__name__ + " " + str(exc)).upper()
        if any(token in upper for token in (
            "401", "403", "API_KEY_INVALID", "PERMISSION_DENIED",
            "API KEY NOT VALID", "BILLING",
        )):
            raise RuntimeError(
                f"Gemini authentication/project access failed ({type(exc).__name__}): {safe_gemini_error(exc)}"
            ) from None
        print(f"Gemini preflight failed for {candidate} (status={_gemini_status(exc)}); trying next model.")

if MODEL_ID is None:
    raise RuntimeError(
        "No configured Gemini model passed preflight. Graph was not started.\n"
        + "\n".join(preflight_errors)
    )

from tradingagents.llm_clients.google_client import GoogleClient

# Install fallback at the provider-client boundary. Original graph agents/prompts/tools
# remain intact; only a transient Gemini transport/rate-limit failure is rerouted.
_original_google_get_llm = GoogleClient.get_llm
_runtime_state = {"active_index": 0}
RUNTIME_MODEL_IDS = [MODEL_ID] + [model for model in MODEL_CANDIDATES if model != MODEL_ID]

def _google_get_llm_with_runtime_fallback(self):
    ids = [self.model] + [model for model in MODEL_CANDIDATES if model != self.model]
    instances = [
        _original_google_get_llm(GoogleClient(model_id, self.base_url, **self.kwargs))
        for model_id in ids
    ]
    return GeminiFallbackRunnable(instances, ids, _runtime_state)

GoogleClient.get_llm = _google_get_llm_with_runtime_fallback

# For FPT, overseas news/insider sources may not cover the ticker. A failed
# enrichment returns an explicit sentinel instead of aborting the entire graph.
from tradingagents.agents import tools as _agent_tools
_original_route_to_vendor = _agent_tools.route_to_vendor
_NEWS_METHODS = {"get_news", "get_global_news", "get_insider_transactions"}

def _safe_news_route(method, *args, **kwargs):
    try:
        return _original_route_to_vendor(method, *args, **kwargs)
    except Exception as exc:
        if method in _NEWS_METHODS:
            print(f"News enrichment unavailable for {method}: {type(exc).__name__}; report as unavailable.")
            return (
                f"DATA_UNAVAILABLE: {method} could not be retrieved from the configured news provider "
                f"({type(exc).__name__}). Do not infer or fabricate news/insider events."
            )
        raise

_agent_tools.route_to_vendor = _safe_news_route

from tradingagents.default_config import DEFAULT_CONFIG
from tradingagents.graph.trading_graph import TradingAgentsGraph

config = deepcopy(DEFAULT_CONFIG)
config["llm_provider"] = "google"
config["quick_think_provider"] = None
config["deep_think_provider"] = None
config["quick_think_llm"] = MODEL_ID
config["deep_think_llm"] = MODEL_ID
config["output_language"] = "Vietnamese"
# Use the runtime fallback immediately instead of spending retries on one overloaded model.
config["llm_max_retries"] = 0
config["data_vendors"] = dict(config.get("data_vendors", {}))
config["data_vendors"]["core_stock_apis"] = "ssi"
config["data_vendors"]["technical_indicators"] = "ssi"
config["data_vendors"]["fundamental_data"] = "vnstock"
config["results_dir"] = str(REPO_DIR / "results")
config["data_cache_dir"] = str(REPO_DIR / ".runtime" / "cache")
config["memory_log_path"] = str(REPO_DIR / ".runtime" / "trading_memory.md")
Path(config["results_dir"]).mkdir(parents=True, exist_ok=True)
Path(config["data_cache_dir"]).mkdir(parents=True, exist_ok=True)
Path(config["memory_log_path"]).parent.mkdir(parents=True, exist_ok=True)

ANALYSIS_DATE = datetime.now(ZoneInfo("Asia/Ho_Chi_Minh")).date().isoformat()
ANALYSTS = ["market", "social", "news", "fundamentals"]
print(f"Starting full TradingAgents graph: FPT / {ANALYSIS_DATE}; initial_model={MODEL_ID}")
print(f"Analysts: {', '.join(ANALYSTS)} -> research/debate -> trader -> risk -> portfolio manager")
print("Sources: SSI OHLCV/indicators; Vnstock VCI statements; configured news/macro sources with unavailable-data guards.")
print("Current-date source contract remains in force; this is not a historical point-in-time backtest.")

ta = TradingAgentsGraph(selected_analysts=ANALYSTS, config=config, debug=False)
final_state, signal = ta.propagate("FPT", ANALYSIS_DATE, asset_type="stock")
report_path = Path(ta.save_reports(final_state, "FPT", html=True)).resolve()
if report_path.is_dir():
    report_dir = report_path
    report_path = report_dir / "complete_report.md"
else:
    report_dir = report_path.parent

if not report_path.is_file() or report_path.stat().st_size <= 0:
    raise RuntimeError(f"TradingAgents finished but complete_report.md is missing/empty: {report_path}")

print("Full TradingAgents graph completed; complete_report.md is non-empty.")
print(f"Active Gemini model at completion: {RUNTIME_MODEL_IDS[min(_runtime_state['active_index'], len(RUNTIME_MODEL_IDS) - 1)]}")
print(f"Signal/rating: {signal}")
print(f"Report size: {report_path.stat().st_size:,} bytes")
print(f"Report directory: {report_dir}")
print("Run only the final cell if you want to package/download this report again.")


In [ ]:
# 4) PACKAGE + DOWNLOAD EXISTING REPORT ONLY.
# This cell never calls Gemini or SSI. Run only this cell to download the report again.
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files as colab_files

def find_existing_fpt_report(candidate=None):
    if candidate is not None:
        p = Path(candidate).expanduser()
        if p.is_file() and (p.name.lower() == "complete_report.md" or p.suffix.lower() == ".md"):
            return p.resolve()
        if p.is_dir():
            expected = p / "complete_report.md"
            if expected.is_file():
                return expected.resolve()

    roots = [
        Path("/root/.tradingagents/logs/reports"),
        Path("/content/TradingAgentsFresh/results/reports"),
        Path("/content/results/reports"),
    ]
    matches = []
    for root in roots:
        if not root.exists():
            continue
        if root.is_file() and root.suffix.lower() == ".md":
            matches.append(root)
        elif root.is_dir():
            matches.extend(
                p for p in root.rglob("complete_report.md")
                if p.is_file() and p.parent.name.startswith("FPT_")
            )
    if not matches:
        raise FileNotFoundError(
            "No existing FPT complete_report.md found in this runtime. "
            "This cell will not rerun AI; the earlier runtime may have been reset."
        )
    return max(set(p.resolve() for p in matches), key=lambda p: p.stat().st_mtime)

candidate = globals().get("report_path")
if candidate is None:
    candidate = globals().get("report_dir")
existing_report = find_existing_fpt_report(candidate)

if existing_report.stat().st_size <= 0:
    raise RuntimeError(f"Report is empty; refusing to create ZIP: {existing_report}")

existing_dir = existing_report.parent
items = sorted(
    p for p in existing_dir.rglob("*")
    if p.is_file() and not p.is_symlink()
)
if existing_report not in items:
    items.append(existing_report)
items = sorted(set(items))
if not items:
    raise RuntimeError(f"No files found for packaging: {existing_dir}")

archive = Path("/content/TradingAgents_FPT_report.zip")
temp_archive = Path(str(archive) + ".tmp")
archive.parent.mkdir(parents=True, exist_ok=True)
temp_archive.unlink(missing_ok=True)

try:
    with ZipFile(temp_archive, "w", compression=ZIP_DEFLATED) as zf:
        for item in items:
            zf.write(item, arcname=f"{existing_dir.name}/{item.relative_to(existing_dir)}")

    with ZipFile(temp_archive, "r") as zf:
        names = zf.namelist()
        corrupt = zf.testzip()
        expected = f"{existing_dir.name}/{existing_report.relative_to(existing_dir)}"
        if not names or corrupt:
            raise RuntimeError(f"ZIP verification failed: empty={not names}; corrupt member={corrupt}")
        if expected not in names or zf.getinfo(expected).file_size <= 0:
            raise RuntimeError("ZIP verification failed: complete_report.md missing or empty")
    if temp_archive.stat().st_size <= 0:
        raise RuntimeError("ZIP archive has zero bytes.")
    temp_archive.replace(archive)
finally:
    temp_archive.unlink(missing_ok=True)

print(f"ZIP VERIFIED: {len(names)} files; {archive.stat().st_size:,} bytes.")
print(f"Report inside ZIP: {expected}; report size={existing_report.stat().st_size:,} bytes.")
for name in names:
    print(" -", name)
print("Starting browser download. This cell makes no Gemini/SSI calls.")
colab_files.download(str(archive))
